## Settings

In [1]:
import hashlib, json, time
from pathlib import Path
import numpy as np
import open3d as o3d
import pycolmap

ROOT = Path.cwd().parents[2]                       # .../Case_Study
SCENE  = "IMG_9175"
SOURCE = "sparse"                                  # "sparse" or "dense"

RUN      = ROOT / "data" / "colmap" / SCENE / "run_full_colab"   # Stage 1, read only
OUT_ROOT = ROOT / "data" / "mesh" / SCENE                        # Stage 2A output
FUSED    = OUT_ROOT / "dense_input" / "fused.ply"                # from Colab
assert (RUN / "sparse_best").exists(), f"Stage 1 run not found: {RUN}"

# Starting points, not tuned values
ROI_SCALE = 1.0                  # crop radius = ROI_SCALE x median camera distance
VOXEL     = 0.0 if SOURCE == "sparse" else 0.01    # SfM units, 0 = off
NB_NEIGHBORS, STD_RATIO = 20, 2.0
NORMAL_KNN = 30
print("open3d", o3d.__version__, "| pycolmap", pycolmap.__version__)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
open3d 0.20.0 | pycolmap 4.2.1


## Cameras and Points

In [2]:
rec = pycolmap.Reconstruction(str(RUN / "sparse_best"))
images = list(rec.images.values())
cams = np.array([im.projection_center() for im in images])

toward = None        # per-point direction to the cameras that saw it (sparse only)
if SOURCE == "sparse":
    pts = list(rec.points3D.values())
    xyz = np.array([p.xyz for p in pts])
    center_of = {i: im.projection_center() for i, im in rec.images.items()}
    toward = np.empty_like(xyz)
    for k, p in enumerate(pts):
        v = np.array([center_of[e.image_id] for e in p.track.elements]) - p.xyz
        toward[k] = (v / np.linalg.norm(v, axis=1, keepdims=True)).sum(axis=0)
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(xyz))
    pcd.colors = o3d.utility.Vector3dVector(np.array([p.color for p in pts]) / 255.0)
elif SOURCE == "dense":
    pcd = o3d.io.read_point_cloud(str(FUSED))
    assert len(pcd.points) > 0, f"Could not read {FUSED}"
    assert pcd.has_normals(), "fused.ply has no normals"
else:
    raise ValueError(SOURCE)

n_loaded = len(pcd.points)
print("registered frames:", rec.num_reg_images(), "| points:", n_loaded,
      "| colors:", pcd.has_colors())

registered frames: 200 | points: 94849 | colors: True


## Crop to subject

In [ ]:
# Subject center = point closest to all camera viewing axes (suits an inward orbit)
d = np.array([im.viewing_direction() for im in images])
Q = np.eye(3) - d[:, :, None] * d[:, None, :]
center = np.linalg.solve(Q.sum(axis=0), np.einsum("nij,nj->i", Q, cams))
radius = ROI_SCALE * np.median(np.linalg.norm(cams - center, axis=1))

keep = np.flatnonzero(np.linalg.norm(np.asarray(pcd.points) - center, axis=1) <= radius)
pcd = pcd.select_by_index(keep)
if toward is not None:
    toward = toward[keep]
n_cropped = len(pcd.points)
print(f"ROI radius {radius:.2f} | kept {n_cropped} of {n_loaded}")
o3d.visualization.draw_geometries([pcd], window_name="cropped")

## Remove Statistical Outliers

In [ ]:
if VOXEL > 0:
    assert toward is None, "voxel downsampling would misalign the sparse view directions"
    pcd = pcd.voxel_down_sample(VOXEL)
n_before = len(pcd.points)
clean, kept = pcd.remove_statistical_outlier(nb_neighbors=NB_NEIGHBORS,
                                             std_ratio=STD_RATIO)
if toward is not None:
    toward = toward[kept]
removed = pcd.select_by_index(kept, invert=True)
removed.paint_uniform_color([1.0, 0.0, 0.0])
pcd = clean
n_clean = len(pcd.points)
print(f"outlier removal: {n_before} -> {n_clean}")
o3d.visualization.draw_geometries([pcd, removed], window_name="red = removed")

## Estimate and Orient Normals

In [ ]:
if SOURCE == "sparse":
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(knn=NORMAL_KNN))
    n = np.array(pcd.normals)
    flip = np.einsum("ij,ij->i", n, toward) < 0
    n[flip] *= -1
    pcd.normals = o3d.utility.Vector3dVector(n)
    print("normals flipped toward their cameras:", int(flip.sum()))
else:
    print("using the normals stored in fused.ply")
o3d.visualization.draw_geometries([pcd], window_name="normals", point_show_normal=True)

## Poisson Reconstruction

In [ ]:
DEPTH = 6            # 6 = quick test. Then try 8, then 9. Rerun cells 6-9.

t = time.time()
raw, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=DEPTH)
densities = np.asarray(densities)
side = 1.1 * pcd.get_axis_aligned_bounding_box().get_max_extent()   # 1.1 = default scale
print(f"depth {DEPTH}: {time.time() - t:.1f} s | finest cell ~{side / 2**DEPTH:.3f} units "
      f"| raw {len(raw.vertices)} vertices, {len(raw.triangles)} triangles")

## Trim and Clean

In [ ]:
DENSITY_Q = 0.05             # fraction of lowest-support vertices to remove
MIN_CLUSTER_FRAC = 0.01      # drop pieces under 1% of the largest piece's triangles

mesh = o3d.geometry.TriangleMesh(raw)                 # copy, so raw stays intact
mesh.remove_vertices_by_mask(densities < np.quantile(densities, DENSITY_Q))
mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()
mesh.remove_duplicated_vertices()
mesh.remove_non_manifold_edges()

labels, counts, _ = mesh.cluster_connected_triangles()
labels, counts = np.asarray(labels), np.asarray(counts)
mesh.remove_triangles_by_mask(counts[labels] < MIN_CLUSTER_FRAC * counts.max())
mesh.remove_unreferenced_vertices()
mesh.compute_vertex_normals()
print(f"trimmed: {len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles")

## Save and Summarize

In [ ]:
def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()[:16]

OUT = OUT_ROOT / f"{SOURCE}_roi{ROI_SCALE}_d{DEPTH}_q{DENSITY_Q}"
OUT.mkdir(parents=True, exist_ok=True)
for name in ("mesh.ply", "mesh.obj"):
    assert o3d.io.write_triangle_mesh(str(OUT / name), mesh), f"failed to write {name}"

inputs = [RUN / "sparse_best" / "images.bin", RUN / "sparse_best" / "points3D.bin"]
if SOURCE == "dense":
    inputs.append(FUSED)

_, pieces, _ = mesh.cluster_connected_triangles()
summary = {
    "scene": SCENE, "source": SOURCE,
    "inputs_sha256_16": {p.relative_to(ROOT).as_posix(): sha(p) for p in inputs},
    "versions": {"open3d": o3d.__version__, "pycolmap": pycolmap.__version__},
    "settings": {"roi_scale": ROI_SCALE, "voxel": VOXEL, "nb_neighbors": NB_NEIGHBORS,
                 "std_ratio": STD_RATIO, "normal_knn": NORMAL_KNN, "depth": DEPTH,
                 "density_quantile": DENSITY_Q, "min_cluster_frac": MIN_CLUSTER_FRAC},
    "points": {"loaded": n_loaded, "cropped": n_cropped, "after_outliers": n_clean},
    "mesh": {"vertices": len(mesh.vertices), "triangles": len(mesh.triangles),
             "connected_pieces": len(pieces),
             "closed_edges": mesh.is_edge_manifold(allow_boundary_edges=False),
             "watertight": mesh.is_watertight()},
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("saved to", OUT)

## View

In [ ]:
o3d.visualization.draw_geometries([mesh], window_name=f"depth {DEPTH}",
                                  mesh_show_back_face=True)